<a href="https://colab.research.google.com/github/AliMahmoud67/FlyRank_Starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliMahmoud67/FlyRank_Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content item for one client on one report date.

**Time window:** I will use February 2026 as the feature window and March 2026 as the subsequent outcome window. This keeps the features in the past relative to the outcome.

**Tables:** `fact_content_daily_performance` (daily GSC metrics, partitioned by month —
used for both the February features and the March label) joined to `dim_content`
(one row per content item — used for `word_count` and `content_created_date`).

In [2]:
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Outcome window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Outcome window: March 2026


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features:**
- `gsc_impressions` — February search visibility.
- `gsc_clicks` — February search traffic.
- `gsc_avg_position` — February average search position.
- `word_count` — content length from `dim_content`.
- `content_age_days` — calculated from `content_created_date` relative to the February decision cutoff.

**Label / proxy:**
- `march_ctr` — future March 2026 CTR, calculated as March `gsc_clicks / gsc_impressions`. This is the future outcome we use to evaluate the content after the February decision cutoff.
- March `gsc_clicks` and `gsc_impressions` — source fields used to calculate the March CTR.

**Context:**
- `client_hash_id` — grouping and joining.
- `content_hash_id` — grouping and joining.
- `report_date` — defining the feature and outcome windows.
- `content_created_date` — used to calculate content age.
- `gsc_data_available` — identifying rows with usable GSC data.

**Excluded:**
- `trend_direction` — derived from trend information and can reveal the outcome.
- `trend_pct` — used to derive `trend_direction`, so it can leak the target.
- `is_declining_label` — directly derived from the trend mechanism and therefore not an independent feature.
- Any March/future performance fields as model features — they are only available after the decision cutoff and would leak future information.

In [4]:
DIM = f"read_parquet('{REL}/dim_content.parquet')"

con.execute(f"""
    SELECT *
    FROM {DIM}
    LIMIT 1
""").df().columns.tolist()

['client_hash_id',
 'content_hash_id',
 'keyword_hash_id',
 'url_hash_id',
 'keyword_char_count',
 'keyword_token_count',
 'url_char_count',
 'content_created_date',
 'content_updated_date',
 'content_type',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'main_intent',
 'backlinks',
 'category_count',
 'keyword_created_date',
 'provider_used',
 'model_used',
 'char_count',
 'word_count',
 'last_optimized_date',
 'optimization_eligible_date',
 'is_published',
 'is_deleted']

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {FEB}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,row_count


In [6]:
con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FEB}
""").df()

,row_count,min_date,max_date
0,7355108,2026-02-01,2026-02-28


In [7]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM {FEB}
""").df()

,total_rows,gsc_available_rows
0,7355108,2621783


### Five-feature frame

The five features are calculated only from information available by the February 2026 decision cutoff.

- `gsc_impressions` — total February GSC impressions. **Available when:** February 2026 closes.
- `gsc_clicks` — total February GSC clicks. **Available when:** February 2026 closes.
- `gsc_avg_position` — impression-weighted February average position. **Available when:** February 2026 closes.
- `word_count` — content word count from `dim_content`. **Available when:** before the decision cutoff.
- `content_age_days` — days since content creation, calculated at the February 2026 cutoff. **Available when:** before the decision cutoff.

In [9]:
feb_features = con.execute(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions) AS gsc_impressions,
        SUM(f.gsc_clicks) AS gsc_clicks,
        SUM(f.gsc_sum_position)
            / NULLIF(SUM(f.gsc_impressions), 0) AS gsc_avg_position,
        d.word_count,
        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-02-28'
        ) AS content_age_days
    FROM {FEB} f
    JOIN {DIM} d
      ON f.client_hash_id = d.client_hash_id
     AND f.content_hash_id = d.content_hash_id
    WHERE f.gsc_data_available IS TRUE
    AND d.content_created_date <= DATE '2026-02-28'
    GROUP BY
        f.client_hash_id,
        f.content_hash_id,
        d.word_count,
        d.content_created_date
    LIMIT 10
""").df()

feb_features

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_age_days
0,client_3ffa76342f366962,content_8127ec118427d857,6.0,0.0,5.666667,3010,31
1,client_3ffa76342f366962,content_065333866c7ab633,20.0,0.0,4.350000,2470,31
2,client_3ffa76342f366962,content_21824d3c6cfc5816,3.0,0.0,4.333333,2620,31
3,client_3ffa76342f366962,content_77189444ffea7d0a,3.0,0.0,11.333333,2715,31
4,client_3ffa76342f366962,content_295c2df25b4bd300,6.0,0.0,5.333333,2692,31
5,client_3ffa76342f366962,content_8ae35855dafb6e51,11.0,0.0,6.181818,3065,31
6,client_3ffa76342f366962,content_f8d4c64da6a26b56,28.0,1.0,4.607143,2513,31
7,client_3ffa76342f366962,content_cc3b2c0222c6b51b,1.0,0.0,38.000000,2824,30
8,client_3ffa76342f366962,content_c43626fdd25399a8,6.0,0.0,24.833333,3132,30
9,client_3ffa76342f366962,content_db624f0521a6dd78,4.0,0.0,7.500000,2967,30


In [10]:
mar_outcomes = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS march_ctr
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
    LIMIT 10
""").df()

mar_outcomes

,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr
0,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.002028
1,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.001418
2,client_62f4a7e64f5e0096,content_df22bda1218f13ff,2099.0,1.0,0.000476
3,client_62f4a7e64f5e0096,content_aa184c1b4ea518e5,288.0,0.0,0.000000
4,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,3535.0,25.0,0.007072
5,client_62f4a7e64f5e0096,content_d7568011c4325a33,1558.0,3.0,0.001926
6,client_62f4a7e64f5e0096,content_e847a4dcc8af3742,2021.0,0.0,0.000000
7,client_62f4a7e64f5e0096,content_85b1be9944e4e19d,1257.0,0.0,0.000000
8,client_62f4a7e64f5e0096,content_b6a42d76effd1906,724.0,1.0,0.001381
9,client_62f4a7e64f5e0096,content_ed474877f5c418f5,161.0,0.0,0.000000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data limitation:** GSC data is not available for all rows. In February 2026, only 2621783 of 7355108 rows had `gsc_data_available = TRUE`. Therefore, the model cannot treat missing GSC data as zero performance, and the usable training population will be smaller than the full February dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


#The data leakage is in w03_feature_leakage_check.ipynb

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.